In [69]:
import os
import platform
import sys
import multiprocessing

def get_ram_gb():
    try:
        import psutil
        return round(psutil.virtual_memory().total / (1024**3), 2)
    except ImportError:
        return "psutil not installed"

print("=== SYSTEM INFORMATION ===")
print(f"OS: {platform.system()} {platform.release()}")
print(f"OS Version: {platform.version()}")
print(f"Machine: {platform.machine()}")
print(f"Processor: {platform.processor()}")
print(f"CPU Cores (logical): {multiprocessing.cpu_count()}")
print(f"Python Version: {sys.version}")
print(f"RAM (GB): {get_ram_gb()}")

=== SYSTEM INFORMATION ===
OS: Linux 6.8.12-11-pve
OS Version: #1 SMP PREEMPT_DYNAMIC PMX 6.8.12-11 (2025-05-22T09:39Z)
Machine: x86_64
Processor: x86_64
CPU Cores (logical): 8
Python Version: 3.12.3 (main, Jan 22 2026, 20:57:42) [GCC 13.3.0]
RAM (GB): 32.0


In [ ]:
print("\n=== GPU INFORMATION ===")

try:
    import torch
    print(f"PyTorch Version: {torch.__version__}")
    print(f"CUDA Available: {torch.cuda.is_available()}")

    if torch.cuda.is_available():
        print(f"GPU Name: {torch.cuda.get_device_name(0)}")
        print(f"GPU Count: {torch.cuda.device_count()}")
        print(f"GPU Total Memory (GB): {round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2)}")
        print(f"CUDA Version (PyTorch): {torch.version.cuda}")
        print(f"GPU Compute Capability: {torch.cuda.get_device_capability(0)}")
except ImportError:
    print("PyTorch not installed")


=== GPU INFORMATION ===
PyTorch Version: 2.11.0+cu128
CUDA Available: True
GPU Name: NVIDIA A100-PCIE-40GB
GPU Count: 1
GPU Total Memory (GB): 39.49
CUDA Version (PyTorch): 12.8
GPU Compute Capability: (8, 0)


# Original Jeepney Routes

## Route Connection and Genetic Algorithm Functions

### Setting Up

In [ ]:
import os
import sys
sys.path.insert(0, os.path.abspath("../src"))

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import geopandas as gpd
import networkx as nx
import shapely
import folium
import geojson
import math
import osmnx as ox
from rtree import index as rtree_index
import pickle
import copy
import utm
import csv

from shapely.ops import unary_union
from shapely.geometry import Polygon, MultiPolygon, LineString, Point
from geopy.distance import geodesic
from shapely.ops import split

from __future__ import absolute_import, division
from math import radians, sin, cos, sqrt, atan2, exp, log
import webbrowser
import random
from scipy.spatial import KDTree
from scipy.spatial.distance import euclidean

import os
import time
from PIL import Image, ImageChops
from selenium import webdriver
from selenium.webdriver.chrome.options import Options

ox.settings.log_console=True
ox.settings.use_cache=True

# Connectivity metric configuration
TARGET_NODE_RATIO = 0.75   # for N_i
CITY_CONNECTIVITY_CONFIG = {
    "Manila": {
        "min_intersections_per_route": 4,
        "min_transpo_stops_per_route": 3,
    },
    "Makati": {
        "min_intersections_per_route": 3,
        "min_transpo_stops_per_route": 2,
    },
    "Mandaluyong": {
        "min_intersections_per_route": 2,
        "min_transpo_stops_per_route": 1,
    },
    "Taguig": {
        "min_intersections_per_route": 3,
        "min_transpo_stops_per_route": 2,
    },
}

# Total potential transport nodes. Set them equal to corresponding GA values
total_potential_transport_nodes_Manila = 464
total_potential_transport_nodes_Makati = 111
total_potential_transport_nodes_Mandaluyong = 45
total_potential_transport_nodes_Taguig = 16

In [ ]:
# For reproducibility. Makes any random sampling deterministic.
SEED = 42 # You can change this value to any integer depending on the run
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED);
np.random.seed(SEED)

In [ ]:
# Schema comes from the shared source now, so what this notebook pickles is
# exactly what the GA reads. stopCandidate stays local: it is only a carrier for
# a stop's coordinates while the routes are being assembled, and it predates
# schema.Stop, which also wants an area.
from genetic_algorithm.schema import Network, Route


class stopCandidate:
    def __init__(self, lat, long, isTranspo, id):
        self.lat = lat
        self.long = long
        self.isTranspo = isTranspo
        self.id = id  # Stop ID

    def getLat(self):
        return self.lat

    def getLong(self):
        return self.long

    def getDegree(self):
        return self.degree


In [74]:
# For units
def degrees_to_meters(angle_degrees):
    return angle_degrees * 6371000 * math.pi / 180

def meters_to_degrees(distance_meters):
    return distance_meters / 6371000 * 180 / math.pi

In [75]:
# Import and Export networks or graphs to pickle
def save_data(networks, path):
    with open(path, 'wb') as f:
        pickle.dump(networks, f)

def load_data(path):
    with open(path, 'rb') as f:
        routes = pickle.load(f)
    return routes

In [ ]:
import re

def process_csv(file_path):
    with open(file_path, mode='r', newline='') as file:
        point_pattern = re.compile(r'POINT \(([-\d.]+) ([-\d.]+)\)')
        reader = csv.DictReader(file)

        # Variables to hold the current route list and all routes list
        current_route_number = None
        current_route_list = []
        all_routes_list = []

        for row in reader:
            wkt = row['WKT']
            route_number = row['Route Number']

            match = point_pattern.match(wkt)
            if match:
                x, y = match.groups()
                x, y = float(x), float(y)
            else:
                x, y = None, None

            # Check if the route number has changed
            if route_number != current_route_number:
                # If there's a current route list, append it to the all routes list
                if current_route_list:
                    all_routes_list.append(current_route_list)

                # Start a new route list
                current_route_list = []
                current_route_number = route_number

            # Append the current row to the current route list
            current_route_list.append([(x, y), route_number])

        # Append the last route list to the all routes list
        if current_route_list:
            all_routes_list.append(current_route_list)

    return all_routes_list

#### Graph and Features

In [ ]:
# GENERATION OF MAIN CITY GRAPH
# IF FIRST TIME RUNNING, RUN THIS CODE TO GENERATE THE GRAPH
def generate_graph():
    mode = 'drive'
    graph = ox.graph_from_place(select_city, network_type = mode) # Generate graph of Metro manila
    ox.save_graphml(graph, city_file) # Save it as a file

def load_graph():
    graph = ox.load_graphml(city_file)

    print("Graph loaded successfully")
    print("NUMBER OF EDGES: ", graph.number_of_edges())
    print("NUMBER OF NODES: ", graph.number_of_nodes())
    print('\n')
    return graph

select_city = "Metro Manila, Philippines"
city_file = '../data/street_networks/Metro Manila.graphml'
#generate_graph()
METROMANILA_GRAPH = load_graph()

select_city = "Manila, Philippines"
city_file = '../data/street_networks/Manila.graphml'
#generate_graph()
MANILA_GRAPH = load_graph()

select_city = "Mandaluyong, Philippines"
city_file = '../data/street_networks/Mandaluyong.graphml'
#generate_graph()
MANDALUYONG_GRAPH = load_graph()

select_city = "Makati, Philippines"
city_file = '../data/street_networks/Makati.graphml'
#generate_graph()
MAKATI_GRAPH = load_graph()

In [ ]:
### For Filtering the roads and other features
# GETTING ROADS AND WATERWAYS

# Get all the roads in Manila
manila_road = ox.graph_to_gdfs(MANILA_GRAPH,nodes=False, edges=True)
filtered_roads = manila_road[manila_road['junction'].isna()]

# Separate roads whose highway types are only one value and those that are more than 1 (lists)
rows_with_lists = filtered_roads[filtered_roads['highway'].apply(lambda x: isinstance(x, list))]
rows_with_strings = filtered_roads[filtered_roads['highway'].apply(lambda x: isinstance(x, str))]

# Allowed Roads to place stops
filter_options = ['primary', 'secondary', 'tertiary', 'trunk', 'unclassified']

# Get all the roads with the allowed road types
filtered_roads_strings_manila = rows_with_strings.loc[rows_with_strings['highway'].isin(filter_options)]

# Total Distance of the allowed road types
TOTAL_NETWORK_ROAD_DISTANCE_MANILA = 0
for index, row in filtered_roads_strings_manila.iterrows():
    TOTAL_NETWORK_ROAD_DISTANCE_MANILA += row['length']

# MAKATI
makati_road = ox.graph_to_gdfs(MAKATI_GRAPH,nodes=False, edges=True)
filtered_roads = makati_road[makati_road['junction'].isna()]

# Separate roads whose highway types are only one value and those that are more than 1 (lists)
rows_with_lists = filtered_roads[filtered_roads['highway'].apply(lambda x: isinstance(x, list))]
rows_with_strings = filtered_roads[filtered_roads['highway'].apply(lambda x: isinstance(x, str))]

# Allowed Roads to place stops
filter_options = ['primary', 'secondary', 'tertiary', 'trunk', 'unclassified']

# Get all the roads with the allowed road types
filtered_roads_strings_makati = rows_with_strings.loc[rows_with_strings['highway'].isin(filter_options)]

# Total Distance of the allowed road types
TOTAL_NETWORK_ROAD_DISTANCE_MAKATI = 0
for index, row in filtered_roads_strings_makati.iterrows():
    TOTAL_NETWORK_ROAD_DISTANCE_MAKATI += row['length']


# MANDALUYONG
mandaluyong_road = ox.graph_to_gdfs(MANDALUYONG_GRAPH,nodes=False, edges=True)
filtered_roads = mandaluyong_road[mandaluyong_road['junction'].isna()]

# Separate roads whose highway types are only one value and those that are more than 1 (lists)
rows_with_lists = filtered_roads[filtered_roads['highway'].apply(lambda x: isinstance(x, list))]
rows_with_strings = filtered_roads[filtered_roads['highway'].apply(lambda x: isinstance(x, str))]

# Allowed Roads to place stops
filter_options = ['primary', 'secondary', 'tertiary', 'trunk', 'unclassified']

# Get all the roads with the allowed road types
filtered_roads_strings_mandaluyong = rows_with_strings.loc[rows_with_strings['highway'].isin(filter_options)]

# Total Distance of the allowed road types
TOTAL_NETWORK_ROAD_DISTANCE_MANDALUYONG = 0
for index, row in filtered_roads_strings_mandaluyong.iterrows():
    TOTAL_NETWORK_ROAD_DISTANCE_MANDALUYONG += row['length']

### Reading Data

In [ ]:
# Reading all points
#
# NOTE: the amenity maps are their own repository, checked in as the submodule
# at data/amenity_maps. If these reads fail the submodule is not populated yet:
#     git submodule update --init data/amenity_maps
from analysis.original_network import amenity_maps_path

manila_points_gdf = gpd.read_file(amenity_maps_path('Jacob-Group', 'Manila City', 'Manila_point.geojson'))
mandaluyong_points_gdf = gpd.read_file(amenity_maps_path('Jacob-Group', 'Mandaluyong City', 'Mandaluyong_point.geojson'))
makati_points_gdf = gpd.read_file(amenity_maps_path('Jacob-Group', 'Makati City', 'Makati_point.geojson'))

manila_transpo_gdf = manila_points_gdf[manila_points_gdf['amenity'] == 'transportation']
mandaluyong_transpo_gdf = mandaluyong_points_gdf[mandaluyong_points_gdf['amenity'] == 'transportation']
makati_transpo_gdf = makati_points_gdf[makati_points_gdf['amenity'] == 'transportation']


In [ ]:
# Index for optimization
manila_idx = rtree_index.Index()
for i, row in manila_transpo_gdf.iterrows():
    manila_idx.insert(i, row['geometry'].bounds)

mandaluyong_idx = rtree_index.Index()
for i, row in mandaluyong_transpo_gdf.iterrows():
    mandaluyong_idx.insert(i, row['geometry'].bounds)

makati_idx = rtree_index.Index()
for i, row in makati_transpo_gdf.iterrows():
    makati_idx.insert(i, row['geometry'].bounds)


In [ ]:
# Function to Extract all the stop points per row
def extract_points(df):
    routes = []
    columns = list(df.columns)

    for index, row in df.iterrows():
        points = []
        for col in columns:
            cell_value = row[col]
            if pd.isna(cell_value):
                continue

            if cell_value.endswith(']'):
                cell_value = cell_value[:-1] + ',' # Remove the last bracket if it's there

            data = eval(cell_value)[0]

            lat = data[0]
            lon = data[1]
            points.append((lat, lon))
        routes.append(points)
    return routes

In [ ]:
# REVISED ROUTES

# MANILA
file_path = '../data/jeepney_routes/Original Data/Formatted-Manila-Routes.csv'
manila_route_stops = process_csv(file_path)

# MAKATI
file_path = '../data/jeepney_routes/Original Data/Existing Jeepney Routes (Makati).csv'
makati_route_stops = process_csv(file_path)

# MANDALUYONG
file_path = '../data/jeepney_routes/Original Data/Existing Jeepney Routes (Mandaluyong).csv'
mandaluyong_route_stops = process_csv(file_path)

### Stop Placement

In [ ]:
# Keep the parts of each route that lie inside the city road graph.
#
# NOTE: this used to cut a route at the first stop outside the graph. Taguig's
# routes run into Pasig and Makati and come back, so that threw away two thirds
# of the network. Keeping the in-city runs instead preserves what is in scope
# without inventing road the jeepney never takes.
from analysis.build_original import clip_to_city


def remove_if_out_of_bounds(city_graph, routes, tolerance):
    segments, dropped, split = clip_to_city(routes, city_graph)
    print(f"dropped {dropped} stops outside the graph; {split} routes split")
    return segments


In [ ]:
# This function checks if there are stops that are transpo stop and labels them as such
def get_close_points(idx, point, max_distance=50):
    close_points = [i for i in idx.intersection((point.x - max_distance, point.y - max_distance, point.x + max_distance, point.y + max_distance))]
    return close_points


def check_transpo_stops(routes, point_gdf, city_idx, max_distance_to_stop):
    transpo_points = []
    non_transpo_points = []
    new_route_list_with_stop_id = []

    stop_id = 0
    for route in routes:
        new_route = []
        for point, name in route:
            long = point[0]
            lat = point[1]
            isTranspo = False

            close_points = get_close_points(city_idx, Point(long, lat))
            for i, row in point_gdf.loc[close_points].iterrows():
                distance_to_point = geodesic((lat, long), (row['y'], row['x'])).meters

                # Check if distance is within 100 meters and it's a transportation amenity
                if distance_to_point <= max_distance_to_stop and row['amenity'] == 'transportation':
                    isTranspo = True
                    break

            new_stop = stopCandidate(lat, long, isTranspo, stop_id)
            new_route.append(new_stop)

            if isTranspo:
                transpo_points.append(new_stop)
            else:
                non_transpo_points.append(new_stop)
            stop_id += 1

        new_route_list_with_stop_id.append(new_route)

    return transpo_points, non_transpo_points, new_route_list_with_stop_id

In [ ]:
# Insert a stop as a node on the road edge nearest to it.
#
# NOTE: this used to cut only one direction of the edge. A two-way road is two
# directed edges, so splitting one left the stop reachable going one way and not
# the other, and the path back looped around the block -- on Taguig that made
# 14km paths between stops 100m apart, and it is why reverse legs come out much
# longer than forward ones. splice_stop cuts both directions.
from analysis.build_original import splice_stop


def add_point_to_graph(point, graph_of_stops, list_of_stops, isTranspo):
    splice_stop(graph_of_stops, point.id, point.lat, point.long, isTranspo)
    list_of_stops.append(point)


In [ ]:
# CREATING STOPS
# It should return a list of coordinates/nodes for stop and a graph of stops
# if residential area, check if the population density

def place_stops_on_roads(transpo_points, non_transpo_points, graph_of_stops, list_of_stops):

    for point in transpo_points:
        add_point_to_graph(point, graph_of_stops, list_of_stops, True)

    for point in non_transpo_points:
        add_point_to_graph(point, graph_of_stops, list_of_stops, False)

In [ ]:
def merge_stops(stops_list):
    stop_dict = {}
    # Sort the stops by latitude and then by longitude
    stops = sorted(stops_list, key=lambda x: (x.lat, x.long), reverse=True)


    grouped_stops = []
    current_group = [stops[0]]

    for i in range(1, len(stops)):
        distance = geodesic((current_group[-1].lat, current_group[-1].long), (stops[i].lat, stops[i].long)).meters
        if distance <= 20: # Distance between the points
            current_group.append(stops[i])
        else:
            grouped_stops.append(current_group)
            current_group = [stops[i]]

    # Last group
    if current_group:
        grouped_stops.append(current_group)


    for group in grouped_stops:

        if len(group) == 1:
            continue

        # Getting the main stop
        main_stop = None
        for stop in group:
            if stop.isTranspo:
                main_stop = stop
                break
        # If no transpo stop, let it be the first stop
        if main_stop == None:
            main_stop = group[0]

        for stop in group:
            if stop.id != main_stop.id:
                stop_dict[stop.id] = main_stop

    return stop_dict

In [ ]:
def change_merged_stops(route_list, non_transpo_points_Manila, transpo_points_Manila, stop_dict):
    transpo_list = []
    non_transpo_list = []
    for route in route_list:
        for stop in route:
            if stop.id in stop_dict:
                index = route.index(stop)
                route[index] = stop_dict[stop.id]

    # Make sure there are no duplicates
    for i, route in enumerate(route_list):
        seen = set()
        unique_stops = []
        for stop in route:
            if stop.id not in seen:
                unique_stops.append(stop)
                seen.add(stop.id)
        route_list[i] = unique_stops

    for stop in non_transpo_points_Manila:
        if stop.id not in stop_dict:
            non_transpo_list.append(stop)

    for stop in transpo_points_Manila:
        if stop.id not in stop_dict:
            transpo_list.append(stop)

    return transpo_list, non_transpo_list

### Route Network Generation

In [89]:
def haversine(lat1, lon1, lat2, lon2):
    # Use geopy's geodesic function to calculate the distance
    distance = geodesic((lat1, lon1), (lat2, lon2)).kilometers
    return distance

In [ ]:
def generate_route_network(routes, graph_of_stops):
    overall_graph = nx.MultiDiGraph() # The route network graph
    route_network = []
    reverse_route_network = []
    num_routes = 0 # Count number of routes

    # Check first if there are paths
    for route in routes:
        prev_stop = None
        for stop in route:
            if prev_stop == None:
                prev_stop = stop
                continue

            if nx.has_path(graph_of_stops, prev_stop.id, stop.id):
                print(f"{prev_stop.id} - {stop.id}")
                prev_stop = stop
            else:
                route_index = routes.index(route)
                stop_index = route.index(prev_stop)
                routes[route_index] = route[:stop_index+1]
                break
        print()

    new_routes = []
    for route in routes:
        if len(route) > 3:
            new_routes.append(route)
    routes = new_routes

    print()
    print("Real route")
    for route in routes:
        route_id = f'{num_routes}-A' # This will be used as a key for the edge
        route_list = []
        prev_stop = None
        totalDistance = 0

        for stop in route:
            if prev_stop == None:
                prev_stop = stop
                continue

            print(f"{prev_stop.id} - {stop.id}")

            path = nx.shortest_path(graph_of_stops, prev_stop.id, stop.id)
            route_list.append(path)

            distance_travelled = 0
            # Get the total distance from point A to point B
            for i in range(len(path)-1):
                node_data = graph_of_stops.nodes[path[i]]
                next_node_data = graph_of_stops.nodes[path[i+1]]
                distance_travelled += haversine(node_data['y'], node_data['x'], next_node_data['y'], next_node_data['x'])

            overall_graph.add_node(prev_stop.id, lat=prev_stop.lat, lon=prev_stop.long, isTranspo=prev_stop.isTranspo) # The origin
            overall_graph.add_node(stop.id, lat=stop.lat, lon=stop.long, isTranspo=stop.isTranspo) # The dest
            overall_graph.add_edge(prev_stop.id, stop.id, key=route_id, road_path = path, distance = distance_travelled) # Add edge
            prev_stop = stop
            totalDistance += distance_travelled


        print()
        new_route_obj = Route(route_list, route_id, totalDistance)
        reverse_route = get_reverse_route(new_route_obj, graph_of_stops, overall_graph)
        route_network.append(new_route_obj)
        reverse_route_network.append(reverse_route)
        num_routes += 1

    return route_network, overall_graph, reverse_route_network



In [ ]:
# Method 1
# Iterate through each node in reverse and get the shortest path

def get_reverse_route(route, graph_of_stops, overall_graph):
    paths = route.route
    reverse_paths = paths[::-1]
    reverse_route_list = []
    reverse_route_id = route.route_id[:-1] + 'B' # Reverse Route ID
    totalDistance = 0

    stop_list = []
    for connection in reverse_paths:
        reverse_route_path = connection[::-1]
        end_node = reverse_route_path[-1]

        # Getting the edges of the start node only for the first connection
        if reverse_paths.index(connection) == 0:
            start_node = reverse_route_path[0] # Start node is critical as it should be the same as the last stop previous connection (If this is not the first connection)

        if not nx.has_path(graph_of_stops, start_node, end_node):
            return None
        else:
            stop_list.append([start_node, end_node])

        start_node = end_node # Next connection start node is the end node of the previous connection

    print(stop_list)
    for connection in stop_list:
        start_node = connection[0]
        end_node = connection[-1]

        reverse_route = nx.shortest_path(graph_of_stops, start_node, end_node, weight='length')
        reverse_route_list.append(reverse_route)

        distance_travelled = 0
        # Get the total distance from point A to point B
        for i in range(len(reverse_route)-1):
            node_data = graph_of_stops.nodes[reverse_route[i]]
            next_node_data = graph_of_stops.nodes[reverse_route[i+1]]
            distance_travelled += haversine(node_data['y'], node_data['x'], next_node_data['y'], next_node_data['x'])

        # Finally, Add the edge
        if reverse_route[0] not in overall_graph:
            overall_graph.add_node(reverse_route[0], **graph_of_stops.nodes[reverse_route[0]])
        if reverse_route[-1] not in overall_graph:
            overall_graph.add_node(reverse_route[-1], **graph_of_stops.nodes[reverse_route[-1]])

        overall_graph.add_edge(reverse_route[0], reverse_route[-1], key=reverse_route_id, road_path = reverse_route, distance = distance_travelled) # Add edge
        totalDistance += distance_travelled

    # print("REVERSE ROUTE DISTANCE: ", totalDistance)
    return Route(reverse_route_list, reverse_route_id, totalDistance)

### Fitness Function

In [ ]:
# NOTE: this notebook used to carry its own copy of the fitness function with
# only five metrics. It now uses the real one, so the baseline and the GA output
# are scored by the same code and redundancy is included.
from genetic_algorithm.fitness import (
    compute_external_connectivity, compute_internal_connectivity,
    compute_redundancy, compute_robustness, compute_vulnerability,
    fitness_capex_cost, fitness_network_efficiency, get_network_coverage,
    normalize_vul_rob)

# The road reference the coverage and redundancy metrics need is derived from
# the city road graph; see analysis.original_network.build_road_reference.
from analysis.original_network import build_road_reference


### Network Analysis Metrics

In [ ]:
# Longest Route
def get_longest_route(network_routes):
    sorted_routes = sorted(network_routes, key=lambda x: x.length, reverse=True)
    longest_route = sorted_routes[0]

    stop_list = [] # Get all the stops
    for connection in longest_route.route:
        if connection[0] not in stop_list:
            stop_list.append(connection[0])
        if connection[-1] not in stop_list:
            stop_list.append(connection[-1])

    return longest_route, stop_list

# Shortest Route
def get_shortest_route(network_routes):
    sorted_routes = sorted(network_routes, key=lambda x: x.length, reverse=True)
    shortest_route = sorted_routes[-1]

    stop_list = [] # Get all the stops
    for connection in shortest_route.route:
        if connection[0] not in stop_list:
            stop_list.append(connection[0])
        if connection[-1] not in stop_list:
            stop_list.append(connection[-1])

    return shortest_route, stop_list

# Average Route Length
def get_average_route_length(network_routes):
    sum_length = sum([n.length for n in network_routes])
    average_length = sum_length / len(network_routes)
    return average_length

def get_longest_path_lengths(snapped_road_network_graph):
    path_lengths = []
    for node in snapped_road_network_graph.nodes():
        lengths = nx.single_source_dijkstra_path_length(snapped_road_network_graph, node, weight='distance')
        list_lengths = [n for target_node, n in lengths.items()]
        path_lengths.append(max(list_lengths))
    return path_lengths

#Network Diameter
def get_network_diameter(network_graph):
    path_lengths = get_longest_path_lengths(network_graph) # Get the sum of all possible
    max_path_length = max(path_lengths)
    return max_path_length

def average_degree(graph):
    degrees = [degree for _, degree in graph.degree()]
    avg_degree = sum(degrees) / len(degrees)
    return avg_degree

def average_number_of_stops_per_route(network_routes):
    num_stops = []
    for route in network_routes:
        seen = set()
        for connection in route.route:
            if connection[0] not in seen:
                seen.add(connection[0])

            if connection[-1] not in seen:
                seen.add(connection[-1])

        num_stops.append(len(seen))

    avg_stops = sum(num_stops) / len(num_stops)
    return avg_stops

In [ ]:
# Node Centrality
def get_node_centrality(network_graph):
    degree_centrality = nx.degree_centrality(network_graph)
    betweenness_centrality = nx.betweenness_centrality(network_graph)
    closeness_centrality = nx.closeness_centrality(network_graph)

    return degree_centrality, betweenness_centrality, closeness_centrality

### Visualizations

In [ ]:
def add_markers(routes, map):
    added = set()
    for route in routes:
        for stop in route:
            if stop.id not in added:
                #popup_text = f"Name: {stop.name}<br>Type: {stop.a_type}<br>Coordinates: {stop.getLat()}, {stop.getLong()}"
                lat = stop.lat
                long = stop.long

                if stop.isTranspo:
                    folium.Marker(location=[lat, long],icon=folium.Icon(color='green')).add_to(map)
                else:
                    folium.Marker(location=[lat, long],icon=folium.Icon(color='blue')).add_to(map)
                added.add(stop.id)

In [ ]:
# This is to visualize the stops
def plot_stops_on_map(stops):
    # Iterate over the nodes in the network
    map_center = (14.599512, 120.984222)
    network_map = folium.Map(location=map_center, zoom_start=10, tiles='openstreetmap')
    for stop in stops:

        if stop.isTranspo:
            marker_color = 'blue'
        else:
            marker_color = 'red'

        folium.Marker(location=[stop.lat, stop.long], popup=f"Transportation: {stop.isTranspo}", icon=folium.Icon(color=marker_color)).add_to(network_map)

    return network_map

In [ ]:
def improved_route_plot(graph, route, color, map, route_id):
    for connection in route.route:
        route_coords = [(graph.nodes[node]['y'], graph.nodes[node]['x']) for node in connection]

        # If this is the first connection, mark the first stop
        if route.route.index(connection) == 0:
            lat = graph.nodes[connection[0]]['y']
            long = graph.nodes[connection[0]]['x']
            isTranspo = graph.nodes[connection[0]]['isTranspo']

            folium.CircleMarker(
            location=[lat, long],
            radius=6,  # Adjust the radius as needed
            popup=f"Transportation: {isTranspo}\n{lat},{long}",
            color=color,
            fill=True,
            fill_color=color,
            fill_opacity=0.7
            ).add_to(map)

        lat = graph.nodes[connection[-1]]['y']
        long = graph.nodes[connection[-1]]['x']
        isTranspo = graph.nodes[connection[-1]]['isTranspo']

        folium.CircleMarker(
        location=[lat, long],
        radius=8,  # The size of the circles
        popup=f"Transportation: {isTranspo}\n{lat},{long}",
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.7
        ).add_to(map)

        folium.PolyLine(route_coords, popup=f"{route_id}", color=color, opacity=0.8).add_to(map)

    return map

In [98]:
def get_network_bounds(network_obj, graph_of_stops, pad_ratio=0.03):
    xs = []
    ys = []

    for route in network_obj.routes:
        for connection in route.route:
            for node in connection:
                xs.append(graph_of_stops.nodes[node]["x"])  # lon
                ys.append(graph_of_stops.nodes[node]["y"])  # lat

    if not xs or not ys:
        return None

    min_x, max_x = min(xs), max(xs)
    min_y, max_y = min(ys), max(ys)

    dx = (max_x - min_x) * pad_ratio
    dy = (max_y - min_y) * pad_ratio

    return [[min_y - dy, min_x - dx], [max_y + dy, max_x + dx]]

In [ ]:
def save_route_map_html(
    network_obj,
    graph_of_stops,
    output_html_path,
    map_center=(14.599512, 120.984222),
    zoom_start=10,
    tiles="openstreetmap",
    zoom_control=False,
    pad_ratio=0.03,
    color_seed=42
):
    """
    Creates a Folium route map, fits bounds to the whole network,
    and saves it as HTML.
    """
    random.seed(color_seed)

    colors = [
        "Red", "Green", "Blue", "Orange", "Purple", "Magenta", "Maroon",
        "Olive", "Navy", "Fuchsia", "Coral", "Indigo", "Violet", "Brown",
        "DarkGreen", "DarkBlue", "DarkRed", "DarkCyan", "DarkMagenta",
        "DarkOrange", "DarkViolet", "DarkTurquoise", "DarkSlateGray",
        "SaddleBrown", "SeaGreen", "MidnightBlue", "ForestGreen", "Sienna",
        "SteelBlue", "RoyalBlue", "SlateBlue", "MediumSeaGreen",
        "MediumSlateBlue", "MediumPurple", "MediumTurquoise", "Crimson",
        "Goldenrod", "Chocolate", "FireBrick", "Peru", "RosyBrown",
        "IndianRed", "SlateGray", "DodgerBlue", "LightSeaGreen",
        "MediumAquamarine", "CadetBlue"
    ]

    m = folium.Map(
        location=map_center,
        zoom_start=zoom_start,
        tiles=tiles,
        zoom_control=zoom_control
    )

    for route in network_obj.routes:
        chosen_color = random.choice(colors)
        improved_route_plot(graph_of_stops, route, chosen_color, m, route.route_id)

    bounds = get_network_bounds(network_obj, graph_of_stops, pad_ratio=pad_ratio)
    if bounds is not None:
        m.fit_bounds(bounds)

    os.makedirs(os.path.dirname(output_html_path), exist_ok=True)
    m.save(output_html_path)
    print(f"Saved HTML map to: {output_html_path}")

## Main

### Creating the Stops

In [ ]:
# File Paths
pikl_filepath = "../data/jeepney_routes/Saved Networks/"
map_filepath = "../data/jeepney_routes/Maps/"

#### Manila Stops

In [ ]:
# MANILA
# Remove all points that are out of bounds
tolerance = 0.0009
manila_route_stops_new = remove_if_out_of_bounds(MANILA_GRAPH, manila_route_stops, tolerance)

# Remove all routes that are less than 3 stops
temp_array = [] # For storing routes that are above 3 stops
for route in manila_route_stops_new:
    if len(route) > 3:
        temp_array.append(route)
manila_route_stops_new = temp_array

In [ ]:
# Getting all stops that are transpos (Stops that are near a transpo point are considered transpo stops)
max_distance_to_transpo_Manila = 30
# Get transpo stops
transpo_points_Manila, non_transpo_points_Manila, complete_route_list_Manila = check_transpo_stops(manila_route_stops_new, manila_points_gdf, manila_idx, max_distance_to_transpo_Manila)

In [ ]:
# Merge stops that are near each other
merged_list = transpo_points_Manila + non_transpo_points_Manila
stop_dict = merge_stops(merged_list)

route_list_Manila = copy.deepcopy(complete_route_list_Manila)
transpo_points_Manila, non_transpo_points_Manila = change_merged_stops(route_list_Manila, non_transpo_points_Manila, transpo_points_Manila, stop_dict)

In [ ]:
print(f"Number of non-transpo stops: {len(non_transpo_points_Manila)}")
print(f"Number of transpo stops: {len(transpo_points_Manila)}")
print(f"Ratio of transpo: {len(transpo_points_Manila) / (len(transpo_points_Manila) + len(non_transpo_points_Manila))}")
#Number of non-transpo stops: 363
#Number of transpo stops: 55
#Ratio of transpo: 0.13157894736842105

Number of non-transpo stops: 363
Number of transpo stops: 55
Ratio of transpo: 0.13157894736842105


In [62]:
# MANILA STOPCANDIDATES (Turn stops into stopCandidates and plot them on the map)
graph_of_stops_Manila = copy.deepcopy(MANILA_GRAPH)
list_of_stops_Manila = []

place_stops_on_roads(transpo_points_Manila, non_transpo_points_Manila, graph_of_stops_Manila, list_of_stops_Manila)

# Visualize the stops
map = plot_stops_on_map(list_of_stops_Manila)
map.save("stop_map_manila.html")

In [ ]:
#Export stops to pickle
file_path = f'{pikl_filepath}stop_list_Manila.pkl'
with open(file_path, 'wb') as f:
    pickle.dump(list_of_stops_Manila, f)

file_path = f'{pikl_filepath}stop_graph_Manila.pkl'
with open(file_path, 'wb') as f:
    pickle.dump(graph_of_stops_Manila, f)

#### Makati

In [64]:
# MAKATI
# Remove all points that are out of bounds
tolerance = 0.9
makati_route_stops_new = remove_if_out_of_bounds(MAKATI_GRAPH, makati_route_stops, tolerance)

# Remove all routes that are less than 3 stops
temp_array = [] # For storing routes that are above 3 stops
for route in makati_route_stops_new:
    if len(route) > 3:
        temp_array.append(route)
makati_route_stops_new = temp_array

In [65]:
# Get transpo stops
max_distance_to_transpo_Makati = 50
transpo_points_Makati, non_transpo_points_Makati, complete_route_list_Makati = check_transpo_stops(makati_route_stops_new, makati_points_gdf, makati_idx, max_distance_to_transpo_Makati)

In [ ]:
# Merge stops in Makati that are near each other

merged_list = transpo_points_Makati + non_transpo_points_Makati
stop_dict = merge_stops(merged_list)

route_list_Makati = copy.deepcopy(complete_route_list_Makati)
transpo_points_Makati, non_transpo_points_Makati = change_merged_stops(route_list_Makati, non_transpo_points_Makati, transpo_points_Makati, stop_dict)

In [67]:
print(f"Number of non-transpo stops: {len(non_transpo_points_Makati)}")
print(f"Number of transpo stops: {len(transpo_points_Makati)}")
print(f"Ratio of transpo: {len(transpo_points_Makati) / (len(transpo_points_Makati) + len(non_transpo_points_Makati))}")

Number of non-transpo stops: 101
Number of transpo stops: 38
Ratio of transpo: 0.2733812949640288


In [68]:
# MAKATI STOPCANDIDATES
graph_of_stops_Makati = copy.deepcopy(MAKATI_GRAPH)
list_of_stops_Makati = []

place_stops_on_roads(transpo_points_Makati, non_transpo_points_Makati, graph_of_stops_Makati, list_of_stops_Makati)

# Visualize the stops
map = plot_stops_on_map(list_of_stops_Makati)
map.save("stop_map_makati.html")

In [ ]:
#Export stops to pickle
file_path = f'{pikl_filepath}stop_list_Makati.pkl'
with open(file_path, 'wb') as f:
    pickle.dump(list_of_stops_Makati, f)

file_path = f'{pikl_filepath}stop_graph_Makati.pkl'
with open(file_path, 'wb') as f:
    pickle.dump(graph_of_stops_Makati, f)

#### Mandaluyong

In [70]:
# MANDALUYONG
# Remove all points that are out of bounds
tolerance = 0.09
mandaluyong_route_stops_new = remove_if_out_of_bounds(MANDALUYONG_GRAPH, mandaluyong_route_stops, tolerance)

# Remove all routes that are less than 3 stops
temp_array = [] # For storing routes that are above 3 stops
for route in mandaluyong_route_stops_new:
    if len(route) > 3:
        temp_array.append(route)
mandaluyong_route_stops_new = temp_array

In [71]:
# Get transpo stops
max_distance_to_transpo_Mandaluyong = 50
transpo_points_Mandaluyong, non_transpo_points_Mandaluyong, complete_route_list_Mandaluyong = check_transpo_stops(mandaluyong_route_stops_new, mandaluyong_points_gdf, mandaluyong_idx, max_distance_to_transpo_Mandaluyong)

In [ ]:
merged_list = transpo_points_Mandaluyong + non_transpo_points_Mandaluyong
stop_dict = merge_stops(merged_list)

route_list_Mandaluyong = copy.deepcopy(complete_route_list_Mandaluyong)
transpo_points_Mandaluyong, non_transpo_points_Mandaluyong = change_merged_stops(route_list_Mandaluyong, non_transpo_points_Mandaluyong, transpo_points_Mandaluyong, stop_dict)

In [73]:
print(f"Number of non-transpo stops: {len(non_transpo_points_Mandaluyong)}")
print(f"Number of transpo stops: {len(transpo_points_Mandaluyong)}")
print(f"Ratio of transpo: {len(transpo_points_Mandaluyong) / (len(transpo_points_Mandaluyong) + len(non_transpo_points_Mandaluyong))}")

Number of non-transpo stops: 60
Number of transpo stops: 10
Ratio of transpo: 0.14285714285714285


In [ ]:
# Test Mandaluyong Map

# Create a map centered at the initial location
map_center = (14.599512, 120.984222) # TEMPORARY WILL ZOOM TO MANILA
m = folium.Map(location=map_center, zoom_start=10, tiles='openstreetmap')

count  = 0
for i, data in mandaluyong_points_gdf.iterrows():
    if data['amenity'] == 'transportation':
        folium.Marker(location=[data['y'], data['x']], popup=f"{data['name']}", icon=folium.Icon(color='blue')).add_to(m)
        count += 1
print(count)

m.save('transpo_mandaluyong.html')

48


In [75]:
# MANDALUYONG STOPCANDIDATES
graph_of_stops_Mandaluyong = copy.deepcopy(MANDALUYONG_GRAPH)
list_of_stops_Mandaluyong = []

place_stops_on_roads(transpo_points_Mandaluyong, non_transpo_points_Mandaluyong, graph_of_stops_Mandaluyong, list_of_stops_Mandaluyong)

# Visualize the stops
map = plot_stops_on_map(list_of_stops_Mandaluyong)
map.save("stop_map_mandaluyong.html")

In [ ]:
#Export stops to pickle
file_path = f'{pikl_filepath}stop_list_Mandaluyong.pkl'
with open(file_path, 'wb') as f:
    pickle.dump(list_of_stops_Mandaluyong, f)

file_path = f'{pikl_filepath}stop_graph_Mandaluyong.pkl'
with open(file_path, 'wb') as f:
    pickle.dump(graph_of_stops_Mandaluyong, f)

### Taguig

Taguig was collected after this notebook last ran, so it has no section of its own
above. It is built here through the same fixed pipeline, which also writes the cache
the EDA notebook reads.

Its routes run well past the city, so only the in-city segments are kept - expect the
printed count of dropped stops to be large.

In [ ]:
from analysis.original_network import build_city_from_csv

Taguig_Route_Network, graph_of_stops_Taguig = build_city_from_csv("Taguig")

print(f"{len(Taguig_Route_Network.routes)} routes, "
      f"{Taguig_Route_Network.graph.number_of_nodes()} stops, "
      f"{Taguig_Route_Network.graph.number_of_edges()} edges")

### Creating the routes

In [ ]:
# Manila
route_network_Manila, route_network_graph_Manila, reverse_route_network_Manila = generate_route_network(route_list_Manila, graph_of_stops_Manila)
Manila_Route_Network = Network(route_network_Manila, route_network_graph_Manila)

In [ ]:
# MAKATI ROUTES
route_network_Makati, route_network_graph_Makati, reverse_route_network_Makati = generate_route_network(route_list_Makati, graph_of_stops_Makati)
Makati_Route_Network = Network(route_network_Makati, route_network_graph_Makati)

In [ ]:
# MANDALUYONG
route_network_Mandaluyong, route_network_graph_Mandaluyong, reverse_route_network_Mandaluyong = generate_route_network(route_list_Mandaluyong, graph_of_stops_Mandaluyong)
Mandaluyong_Route_Network = Network(route_network_Mandaluyong, route_network_graph_Mandaluyong)

### Import Existing Stops and Networks

In [101]:
# Stops
with open(f'{pikl_filepath}stop_graph_Manila.pkl', 'rb') as f:
    graph_of_stops_Manila = pickle.load(f)
with open(f'{pikl_filepath}stop_graph_Makati.pkl', 'rb') as f:
    graph_of_stops_Makati = pickle.load(f)
with open(f'{pikl_filepath}stop_graph_Mandaluyong.pkl', 'rb') as f:
    graph_of_stops_Mandaluyong = pickle.load(f)

# Route Networks
Manila_Route_Network = load_data(f'{pikl_filepath}Manila Route Network.pikl')
Makati_Route_Network = load_data(f'{pikl_filepath}Makati Route Network.pikl')
Mandaluyong_Route_Network = load_data(f'{pikl_filepath}Mandaluyong Route Network.pikl')

### Exporting Route Networks

In [ ]:
#pikl files

save_data(Manila_Route_Network, f'{pikl_filepath}Manila Route Network.pikl')

save_data(Makati_Route_Network, f'{pikl_filepath}Makati Route Network.pikl')

save_data(Mandaluyong_Route_Network, f'{pikl_filepath}Mandaluyong Route Network.pikl')

NameError: name 'export_networks' is not defined

In [102]:
save_route_map_html(
    Manila_Route_Network,
    graph_of_stops_Manila,
    f"{map_filepath}Manila Route Map.html"
)

save_route_map_html(
    Makati_Route_Network,
    graph_of_stops_Makati,
    f"{map_filepath}Makati Route Map.html"
)

save_route_map_html(
    Mandaluyong_Route_Network,
    graph_of_stops_Mandaluyong,
    f"{map_filepath}Mandaluyong Route Map.html"
)

Saved HTML map to: Original Jeepney Routes/Maps/Manila Route Map.html
Saved HTML map to: Original Jeepney Routes/Maps/Makati Route Map.html
Saved HTML map to: Original Jeepney Routes/Maps/Mandaluyong Route Map.html


### Network Metrics Result

In [204]:
def print_city_statistics(
    network,
    avg_betweenness,
    avg_closeness,
    longest_route,
    long_stops,
    shortest_route,
    short_stops,
    avg_route_length,
    avg_degree,
    avg_stops,
    total_road_distance,
    graph_of_stops,
    total_potential_transport_nodes,
    min_intersections_per_route,
    min_transpo_stops_per_route,
    weight_coverage,
    weight_internal_connectivity,
    weight_external_connectivity,
    weight_vulnerability,
    weight_robustness,
):
    fitness_score, components = compute_fitness_score(
        network.graph,
        weight_coverage,
        weight_internal_connectivity,
        weight_external_connectivity,
        weight_vulnerability,
        weight_robustness,
        total_road_distance,
        graph_of_stops,
        total_potential_transport_nodes,
        min_intersections_per_route,
        min_transpo_stops_per_route
    )

    print(f"Coverage Score: {components['Coverage Raw']:.4f}")
    print(f"Internal Connectivity Score: {components['Internal Connectivity Raw']:.4f}")
    print(f"External Connectivity Score: {components['External Connectivity Raw']:.4f}")
    print(f"Vulnerability Score: {components['Vulnerability Raw']:.4f}")
    print(f"Robustness Score: {components['Robustness Raw']:.4f}")
    print(f"Fitness Score: {fitness_score:.4f}")
    print(f"Total Routes: {len(network.routes)}")
    print(f"Total Stops: {len(network.graph.nodes(data=True))}")
    print(f"Total Network Nodes: {components['Total Network Nodes']}")
    print(f"Total Potential Transport Nodes: {components['Total Potential Transport Nodes']}")
    print(f"Shortest Route: {shortest_route:.4f}")
    print(f"Longest Route: {longest_route:.4f}")
    print(f"Average Route Length: {avg_route_length:.4f}")
    print(f"Average Degree: {avg_degree:.4f}")
    print(f"Average Number of Stops per Route: {avg_stops:.4f}")
    print(f"Diameter: {get_network_diameter(network.graph):.4f}")
    print(f"Max Betweenness Centrality: {avg_betweenness:.4f}")
    print(f"Max Closeness Centrality: {avg_closeness:.4f}")
    print(f"n_i: {components['n_i']}")
    print(f"N_i: {components['N_i']}")
    print(f"r_i: {components['r_i']}")
    print(f"R_i: {components['R_i']}")
    print(f"Min Intersections Per Route: {components['Min Intersections Per Route']}")
    print(f"n_e: {components['n_e']}")
    print(f"N_e: {components['N_e']}")
    print(f"r_e: {components['r_e']}")
    print(f"R_e: {components['R_e']}")
    print(f"Min Transpo Stops Per Route: {components['Min Transpo Stops Per Route']}")
    print(f"Target Node Ratio: {components['Target Node Ratio']:.2f}")
    print(f"q_c Vulnerability: {components['q_c Vulnerability']:.4f}")
    print(f"q_c Robustness: {components['q_c Robustness']:.4f}")
    print("")

In [ ]:
_, betweenness_centrality_Manila, closeness_centrality_Manila = get_node_centrality(Manila_Route_Network.graph)
# avg_betweenness_centrality_Manila = sum(betweenness_centrality_Manila.values()) / len(betweenness_centrality_Manila)
# avg_closeness_centrality_Manila = sum(closeness_centrality_Manila.values()) / len(closeness_centrality_Manila)
max_betweenness_centrality_Manila = max(betweenness_centrality_Manila.values())
max_closeness_centrality_Manila = max(closeness_centrality_Manila.values())
longest_route_Manila, long_route_stops_Manila = get_longest_route(Manila_Route_Network.routes)
shortest_route_Manila, short_route_stops_Manila = get_shortest_route(Manila_Route_Network.routes)
avg_route_length_Manila = get_average_route_length(Manila_Route_Network.routes)
avg_degree_Manila = average_degree(Manila_Route_Network.graph)
avg_number_stops_Manila = average_number_of_stops_per_route(Manila_Route_Network.routes)
manila_connectivity_config = CITY_CONNECTIVITY_CONFIG["Manila"]

manila_weights = {
    "weight_coverage": 0.20,
    "weight_internal_connectivity": 0.20,
    "weight_external_connectivity": 0.20,
    "weight_vulnerability": 0.20,
    "weight_robustness": 0.20
}

print("Manila Existing Jeepney Network Statistics")
print_city_statistics(
    Manila_Route_Network,
    max_betweenness_centrality_Manila,
    max_closeness_centrality_Manila,
    longest_route_Manila.length,
    long_route_stops_Manila,
    shortest_route_Manila.length,
    short_route_stops_Manila,
    avg_route_length_Manila,
    avg_degree_Manila,
    avg_number_stops_Manila,
    TOTAL_NETWORK_ROAD_DISTANCE_MANILA,
    graph_of_stops_Manila,
    total_potential_transport_nodes_Manila,
    manila_connectivity_config["min_intersections_per_route"],
    manila_connectivity_config["min_transpo_stops_per_route"],
    #route_list_Manila, # This is for the route list that will be used for the coverage score
    manila_weights["weight_coverage"],
    manila_weights["weight_internal_connectivity"],
    manila_weights["weight_external_connectivity"],
    manila_weights["weight_vulnerability"],
    manila_weights["weight_robustness"]
)

Manila Existing Jeepney Network Statistics
Coverage Score: 0.4515
Internal Connectivity Score: 0.7306
External Connectivity Score: 0.1786
Vulnerability Score: 0.3504
Robustness Score: 0.6590
Fitness Score: 0.4740
Total Routes: 88
Total Stops: 394
Total Network Nodes: 394
Total Potential Transport Nodes: 464
Shortest Route: 1.0571
Longest Route: 37.1687
Average Route Length: 14.3749
Average Degree: 9.4213
Average Number of Stops per Route: 11.6477
Diameter: 38.7395
Max Betweenness Centrality: 0.2237
Max Closeness Centrality: 0.1307
n_i: 183
N_i: 295
r_i: 74
R_i: 88
Min Intersections Per Route: 4
n_e: 55
N_e: 464
r_e: 21
R_e: 88
Min Transpo Stops Per Route: 3
Target Node Ratio: 0.75
q_c Vulnerability: 0.0990
q_c Robustness: 0.0990



In [ ]:
_, betweenness_centrality_Makati, closeness_centrality_Makati = get_node_centrality(Makati_Route_Network.graph)
# avg_betweenness_centrality_Makati = sum(betweenness_centrality_Makati.values()) / len(betweenness_centrality_Makati)
# avg_closeness_centrality_Makati = sum(closeness_centrality_Makati.values()) / len(closeness_centrality_Makati)
max_betweenness_centrality_Makati = max(betweenness_centrality_Makati.values())
max_closeness_centrality_Makati = max(closeness_centrality_Makati.values())
longest_route_Makati, long_route_stops_Makati = get_longest_route(Makati_Route_Network.routes)
shortest_route_Makati, short_route_stops_Makati = get_shortest_route(Makati_Route_Network.routes)
avg_route_length_Makati = get_average_route_length(Makati_Route_Network.routes)
avg_degree_Makati = average_degree(Makati_Route_Network.graph)
avg_number_stops_Makati = average_number_of_stops_per_route(Makati_Route_Network.routes)
makati_connectivity_config = CITY_CONNECTIVITY_CONFIG["Makati"]

makati_weights = {
    "weight_coverage": 0.20,
    "weight_internal_connectivity": 0.20,
    "weight_external_connectivity": 0.20,
    "weight_vulnerability": 0.20,
    "weight_robustness": 0.20
}

print("Makati Existing Jeepney Network Statistics")
print_city_statistics(
    Makati_Route_Network,
    max_betweenness_centrality_Makati,
    max_closeness_centrality_Makati,
    longest_route_Makati.length,
    long_route_stops_Makati,
    shortest_route_Makati.length,
    short_route_stops_Makati,
    avg_route_length_Makati,
    avg_degree_Makati,
    avg_number_stops_Makati,
    TOTAL_NETWORK_ROAD_DISTANCE_MAKATI,
    graph_of_stops_Makati,
    total_potential_transport_nodes_Makati,
    makati_connectivity_config["min_intersections_per_route"],
    makati_connectivity_config["min_transpo_stops_per_route"],
    # route_list_Makati, # This is for the route list that will be used for the coverage score
    makati_weights["weight_coverage"],
    makati_weights["weight_internal_connectivity"],
    makati_weights["weight_external_connectivity"],
    makati_weights["weight_vulnerability"],
    makati_weights["weight_robustness"]
)

Makati Existing Jeepney Network Statistics
Coverage Score: 0.3030
Internal Connectivity Score: 0.7573
External Connectivity Score: 0.3912
Vulnerability Score: 0.6763
Robustness Score: 0.9098
Fitness Score: 0.6075
Total Routes: 25
Total Stops: 139
Total Network Nodes: 139
Total Potential Transport Nodes: 111
Shortest Route: 1.0147
Longest Route: 11.8452
Average Route Length: 4.7568
Average Degree: 6.1583
Average Number of Stops per Route: 9.5600
Diameter: 22.3997
Max Betweenness Centrality: 0.4301
Max Closeness Centrality: 0.1293
n_i: 66
N_i: 104
r_i: 22
R_i: 25
Min Intersections Per Route: 3
n_e: 38
N_e: 111
r_e: 11
R_e: 25
Min Transpo Stops Per Route: 2
Target Node Ratio: 0.75
q_c Vulnerability: 0.0935
q_c Robustness: 0.0935



In [ ]:
_, betweenness_centrality_Mandaluyong, closeness_centrality_Mandaluyong = get_node_centrality(Mandaluyong_Route_Network.graph)
# avg_betweenness_centrality_Mandaluyong = sum(betweenness_centrality_Mandaluyong.values()) / len(betweenness_centrality_Mandaluyong)
# avg_closeness_centrality_Mandaluyong = sum(closeness_centrality_Mandaluyong.values()) / len(closeness_centrality_Mandaluyong)
max_betweenness_centrality_Mandaluyong = max(betweenness_centrality_Mandaluyong.values())
max_closeness_centrality_Mandaluyong = max(closeness_centrality_Mandaluyong.values())
longest_route_Mandaluyong, long_route_stops_Mandaluyong = get_longest_route(Mandaluyong_Route_Network.routes)
shortest_route_Mandaluyong, short_route_stops_Mandaluyong = get_shortest_route(Mandaluyong_Route_Network.routes)
avg_route_length_Mandaluyong = get_average_route_length(Mandaluyong_Route_Network.routes)
avg_degree_Mandaluyong = average_degree(Mandaluyong_Route_Network.graph)
avg_number_stops_Mandaluyong = average_number_of_stops_per_route(Mandaluyong_Route_Network.routes)
mandaluyong_connectivity_config = CITY_CONNECTIVITY_CONFIG["Mandaluyong"]

mandaluyong_weights = {
    "weight_coverage": 0.20,
    "weight_internal_connectivity": 0.20,
    "weight_external_connectivity": 0.20,
    "weight_vulnerability": 0.20,
    "weight_robustness": 0.20
}


print("Mandaluyong Existing Jeepney Network Statistics")
print_city_statistics(
    Mandaluyong_Route_Network,
    max_betweenness_centrality_Mandaluyong,
    max_closeness_centrality_Mandaluyong,
    longest_route_Mandaluyong.length,
    long_route_stops_Mandaluyong,
    shortest_route_Mandaluyong.length,
    short_route_stops_Mandaluyong,
    avg_route_length_Mandaluyong,
    avg_degree_Mandaluyong,
    avg_number_stops_Mandaluyong,
    TOTAL_NETWORK_ROAD_DISTANCE_MANDALUYONG,
    graph_of_stops_Mandaluyong,
    total_potential_transport_nodes_Mandaluyong,
    mandaluyong_connectivity_config["min_intersections_per_route"],
    mandaluyong_connectivity_config["min_transpo_stops_per_route"],
    # route_list_Mandaluyong, # This is for the route list that will be used for the coverage score
    mandaluyong_weights["weight_coverage"],
    mandaluyong_weights["weight_internal_connectivity"],
    mandaluyong_weights["weight_external_connectivity"],
    mandaluyong_weights["weight_vulnerability"],
    mandaluyong_weights["weight_robustness"]
)

Mandaluyong Existing Jeepney Network Statistics
Coverage Score: 0.1955
Internal Connectivity Score: 0.6848
External Connectivity Score: 0.6111
Vulnerability Score: 0.5591
Robustness Score: 0.8495
Fitness Score: 0.5800
Total Routes: 8
Total Stops: 62
Total Network Nodes: 62
Total Potential Transport Nodes: 45
Shortest Route: 3.5106
Longest Route: 9.9029
Average Route Length: 5.6606
Average Degree: 5.6774
Average Number of Stops per Route: 12.2500
Diameter: 13.1033
Max Betweenness Centrality: 0.4111
Max Closeness Centrality: 0.1910
n_i: 17
N_i: 46
r_i: 8
R_i: 8
Min Intersections Per Route: 2
n_e: 10
N_e: 45
r_e: 8
R_e: 8
Min Transpo Stops Per Route: 1
Target Node Ratio: 0.75
q_c Vulnerability: 0.0968
q_c Robustness: 0.0968



### Export CSV

In [ ]:
with open('../data/jeepney_routes/Makati_routes.csv', mode='w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(['lat', 'long', 'route id'])

    for route in Makati_Route_Network.routes:
        for connection in route.route:
            if route.route.index(connection) == 0:
                lat = graph_of_stops_Makati.nodes[connection[0]]['y']
                long = graph_of_stops_Makati.nodes[connection[0]]['x']
                writer.writerow([lat, long, route.route_id])

            lat = graph_of_stops_Makati.nodes[connection[-1]]['y']
            long = graph_of_stops_Makati.nodes[connection[-1]]['x']
            writer.writerow([lat, long, route.route_id])

print("Data exported successfully.")

Data exported successfully.


In [ ]:
with open('../data/jeepney_routes/Mandaluyong_routes.csv', mode='w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(['lat', 'long', 'route id'])

    for route in Mandaluyong_Route_Network.routes:
        for connection in route.route:
            if route.route.index(connection) == 0:
                lat = graph_of_stops_Mandaluyong.nodes[connection[0]]['y']
                long = graph_of_stops_Mandaluyong.nodes[connection[0]]['x']
                writer.writerow([lat, long, route.route_id])

            lat = graph_of_stops_Mandaluyong.nodes[connection[-1]]['y']
            long = graph_of_stops_Mandaluyong.nodes[connection[-1]]['x']
            writer.writerow([lat, long, route.route_id])

print("Data exported successfully.")

Data exported successfully.
